# Turkish-CBE – Revizyon: stepwise (structured reasoning) kolu

**Kullanım:** Çalışma zamanı → *Çalışma zamanı türünü değiştir* → **A100 GPU** (L4 de olur, daha yavaş). Sonra **Çalışma zamanı → Tümünü çalıştır**.

Defter: repoyu klonlar, takip manifestini yeniden üretip SHA-256 ile doğrular, Ollama'yı kurar, `qwen2.5:14b` modelini indirir (digest 7cdf5a01… olmalı), 2025 istemi çalıştırır ve sonucu Google Drive'a yazar. Oturum kesilirse **Tümünü çalıştır** yeniden; tamamlanan istemler atlanır.

Bitince indirilen **`followup_summary.json`** ve **`qwen2-5-14b_structured.jsonl`** dosyalarını Claude'a yükleyin.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT='/content/drive/MyDrive/turkish-cbe-followup'
import os; os.makedirs(OUT, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d turkish-cbe || git clone --depth 1 https://github.com/okanss/turkish-cbe.git
%cd /content/turkish-cbe

In [ ]:
%%writefile scripts/build_followup_manifest.py
"""Build the targeted follow-up subset (revision R1, Reviewer 2 comment 5a).

Subset of the frozen P1 expanded manifest: all nine relations, native_shared
names, EN canonical + TR canonical + TR P3, n in {4, 16, 64}. Rows are copied
verbatim (prompt text and SHA-256 unchanged), so the direct arm can be read
from the existing sweep records and only the structured-reasoning arm is new.
"""
import json, sys
from pathlib import Path
SRC = Path("data/manifests/p1_expanded_manifest.jsonl")
DST = Path("data/manifests/followup_reasoning_manifest.jsonl")
KEEP = {("en", "canonical"), ("tr", "canonical"), ("tr", "P3")}
SIZES = {4, 16, 64}
rows = [json.loads(l) for l in SRC.open(encoding="utf-8")]
sub = [r for r in rows if r["name_control"] == "native_shared"
       and (r["language"], r["condition"]) in KEEP and r["n"] in SIZES]
# keep only query cells that have all three conditions (fully matched)
from collections import defaultdict
cells = defaultdict(set)
for r in sub:
    cells[(r["query_instance_id"], r["relation_id"])].add((r["language"], r["condition"]))
sub = [r for r in sub if cells[(r["query_instance_id"], r["relation_id"])] == KEEP]
sub.sort(key=lambda r: r["prompt_id"])
if DST.exists():
    sys.exit(f"{DST} exists (write-once); delete it deliberately to rebuild")
with DST.open("w", encoding="utf-8") as f:
    for r in sub:
        f.write(json.dumps(r, ensure_ascii=False, sort_keys=True) + "\n")
print(f"{len(sub)} prompts, {len({r['latent_instance_id'] for r in sub})} latent orders -> {DST}")


In [ ]:
%%writefile src/inference/ollama_reasoning_runner.py
"""Local Ollama runner for the revision follow-up (explicit reasoning arm).

Identical to ``inference.ollama_runner`` (same Ollama endpoint, options
recording, checkpoint/resume, pause file) except that messages and parsing
come from the versioned reasoning interventions:
``reasoning.prompts.make_messages(row, strategy)`` (frozen problem frame,
re-verified SHA-256; only the system message and final instruction change)
and ``reasoning.parser.parse_answer`` (one last-line ``FINAL: EVET/HAYIR`` or
``FINAL: YES/NO`` marker; truncation is always invalid).

Usage (structured arm; the direct arm already exists in results/p1/):
    PYTHONPATH=src python3 -m inference.ollama_reasoning_runner \
        --manifest data/manifests/followup_reasoning_manifest.jsonl \
        --model qwen2.5:14b --strategy structured_reasoning \
        --output results/followup/qwen2-5-14b_structured.jsonl
"""

from __future__ import annotations

import argparse
import datetime
import hashlib
import json
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

from reasoning.parser import parse_answer
from reasoning.prompts import make_messages

OLLAMA_URL = "http://localhost:11434"
RUNNER_VERSION = "followup-reasoning-0.1.0"

METADATA_FIELDS = (
    "prompt_id", "latent_instance_id", "query_instance_id", "grid_query_id",
    "split", "language", "condition", "relation_id", "relation_family", "n",
    "name_control", "query_type", "query_distance", "query_direction",
    "gold_label", "sampling_stratum")


def _utc() -> str:
    return datetime.datetime.now(datetime.UTC).isoformat(timespec="milliseconds")


def _post(path: str, payload: dict, timeout: float) -> dict:
    data = json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(OLLAMA_URL + path, data=data,
                                     headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read())


def _get(path: str, timeout: float = 10) -> dict:
    with urllib.request.urlopen(OLLAMA_URL + path, timeout=timeout) as response:
        return json.loads(response.read())


def model_digest(model: str) -> dict:
    for entry in _get("/api/tags")["models"]:
        if entry["name"] == model:
            return {"digest": entry["digest"],
                    "quantization": entry.get("details", {}).get("quantization_level"),
                    "parameter_size": entry.get("details", {}).get("parameter_size"),
                    "weights_bytes": entry["size"]}
    raise ValueError(f"model {model!r} is not installed in the local Ollama store")


def load_manifest(path: Path) -> list[dict]:
    rows = []
    seen = set()
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        missing = [f for f in METADATA_FIELDS if f not in row]
        if missing or "prompt" not in row or "prompt_sha256" not in row:
            raise ValueError(f"manifest row missing fields: {missing or ['prompt']}")
        if row["prompt_id"] in seen:
            raise ValueError("duplicate prompt_id in manifest: " + str(row["prompt_id"]))
        seen.add(row["prompt_id"])
        rows.append(row)
    if not rows:
        raise ValueError("empty manifest")
    return rows


def completed_ids(output: Path) -> set:
    done = set()
    if output.exists():
        for line in output.read_text(encoding="utf-8").splitlines():
            if line.strip():
                record = json.loads(line)
                if record.get("record_kind") == "inference":
                    done.add(record["prompt_id"])
    return done


def run_one(model: str, row: dict, options: dict, timeout: float, retries: int,
            strategy: str = "structured_reasoning") -> dict:
    built = make_messages(row, strategy)  # re-verifies the frozen prompt SHA-256
    payload = {"model": model, "messages": built["messages"], "stream": False,
               "options": options}
    attempts, response, error = 0, None, None
    started = time.monotonic()
    while attempts <= retries:
        attempts += 1
        try:
            response = _post("/api/chat", payload, timeout)
            break
        except (urllib.error.URLError, TimeoutError, OSError, ValueError) as exc:
            error = f"{type(exc).__name__}: {exc}"
            time.sleep(min(2 ** attempts, 30))
    wall = time.monotonic() - started
    record = {
        "record_kind": "inference", "runner_version": RUNNER_VERSION,
        "endpoint": "/api/chat", "timestamp": _utc(),
        **{field: row[field] for field in METADATA_FIELDS},
        "prompt_sha256": row["prompt_sha256"],
        "messages_sha256": built["messages_sha256"],
        "prompt_version": built["prompt_version"], "strategy": strategy,
        "model": model, "requested_options": options,
        "attempts": attempts, "transport_error": None if response else error,
        "wall_seconds": round(wall, 4),
    }
    if response is None:
        record.update(raw_response=None, parsed=None, correct=None,
                      parse_valid=False, failure="transport_failed_after_retries")
        return record
    text = response.get("message", {}).get("content", "")
    done_reason = response.get("done_reason")
    truncated = done_reason == "length"
    context_overflow = (type(response.get("prompt_eval_count")) is int
                        and response["prompt_eval_count"] >= options.get("num_ctx", 0) - 8)
    parsed = parse_answer(text, row["language"], strategy, truncated=truncated,
                          finish_reason=done_reason)
    prediction = parsed.get("prediction")
    record.update(
        raw_response=response, raw_text=text, parsed=parsed,
        parse_valid=bool(parsed.get("valid")),
        prediction=prediction,
        correct=(prediction == row["gold_label"]) if parsed.get("valid") else False,
        done_reason=done_reason, output_truncated=truncated,
        possible_context_truncation=bool(context_overflow),
        prompt_tokens=response.get("prompt_eval_count"),
        output_tokens=response.get("eval_count"),
        prompt_eval_seconds=(response.get("prompt_eval_duration") or 0) / 1e9,
        generation_seconds=(response.get("eval_duration") or 0) / 1e9,
        total_seconds=(response.get("total_duration") or 0) / 1e9,
        failure=None)
    return record


def main(argv=None) -> int:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--manifest", type=Path, required=True)
    parser.add_argument("--model", required=True)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--strategy", default="structured_reasoning",
                        choices=["structured_reasoning", "zero_shot_reasoning", "direct"])
    parser.add_argument("--num-ctx", type=int, default=8192)
    parser.add_argument("--num-predict", type=int, default=4096)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--timeout", type=float, default=300.0)
    parser.add_argument("--retries", type=int, default=3)
    parser.add_argument("--limit", type=int, default=None,
                        help="stop after N new calls (benchmark/smoke use)")
    parser.add_argument("--progress-every", type=int, default=25)
    parser.add_argument("--pause-file", type=Path, default=None,
                        help="if this file exists, finish the current call, "
                             "checkpoint, and exit with code 75 (resumable)")
    args = parser.parse_args(argv)

    rows = load_manifest(args.manifest)
    digest = model_digest(args.model)
    done = completed_ids(args.output)
    pending = [row for row in rows if row["prompt_id"] not in done]
    if args.limit is not None:
        pending = pending[: args.limit]
    manifest_sha = hashlib.sha256(args.manifest.read_bytes()).hexdigest()
    options = {"temperature": 0, "top_k": 1, "seed": args.seed,
               "num_ctx": args.num_ctx, "num_predict": args.num_predict}
    print(json.dumps({"model": args.model, **digest, "manifest_rows": len(rows),
                      "already_completed": len(done), "to_run": len(pending),
                      "options": options}), flush=True)
    args.output.parent.mkdir(parents=True, exist_ok=True)
    header = {"record_kind": "run_header", "runner_version": RUNNER_VERSION,
              "timestamp": _utc(), "model": args.model, **digest,
              "manifest": str(args.manifest), "manifest_sha256": manifest_sha,
              "requested_options": options, "strategy": args.strategy,
              "arm_label": "revision_followup_reasoning; supplemental quantized arm; exploratory"}
    with args.output.open("a", encoding="utf-8") as sink:
        sink.write(json.dumps(header, ensure_ascii=False) + "\n")
        sink.flush()
        started = time.monotonic()
        for index, row in enumerate(pending, 1):
            if args.pause_file is not None and args.pause_file.exists():
                print(f"[{_utc()}] pause requested; {index - 1} new calls written, "
                      "checkpoint is consistent — exiting 75", flush=True)
                return 75
            record = run_one(args.model, row, options, args.timeout, args.retries, args.strategy)
            sink.write(json.dumps(record, ensure_ascii=False) + "\n")
            sink.flush()
            if index % args.progress_every == 0 or index == len(pending):
                rate = index / (time.monotonic() - started)
                remaining = (len(pending) - index) / rate if rate else float("inf")
                print(f"[{_utc()}] {args.model}: {index}/{len(pending)} "
                      f"({rate:.2f} calls/s, ~{remaining/60:.1f} min left)",
                      flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile scripts/analyze_followup.py
"""Analyze the revision follow-up (Table 14): direct vs. structured reasoning.

Direct-arm records are read from the existing sweep file (identical messages);
structured-arm records from the follow-up run. All-planned estimand (invalid =
wrong), latent-order cluster bootstrap from analysis.statistics (2000 reps,
size strata), exactly as for the main contrasts.

    PYTHONPATH=src python3 scripts/analyze_followup.py \
        --direct results/p1/qwen2-5-14b.jsonl \
        --structured results/followup/qwen2-5-14b_structured.jsonl
"""
import argparse, json, sys
from pathlib import Path
sys.path.insert(0, "src")
from analysis.p1_statistics import _pairs
from analysis.statistics import bootstrap

MAN = Path("data/manifests/followup_reasoning_manifest.jsonl")

def load(path, ids):
    recs = {}
    for line in open(path, encoding="utf-8"):
        x = json.loads(line)
        if x.get("record_kind") == "inference" and x["prompt_id"] in ids:
            recs[x["prompt_id"]] = x
    return recs

def allplanned(rows):
    c0 = sum(1 for r in rows if r["original"] is not None and r["original"] == r["gold"])
    c1 = sum(1 for r in rows if r["equivalent"] is not None and r["equivalent"] == r["gold"])
    return {"d": (c1 - c0) / len(rows)}

def arm(name, recs, man):
    out = {"arm": name, "n_records": len(recs), "n_planned": len(man)}
    missing = len(man) - len(recs)
    inv = sum(1 for p in man if p["prompt_id"] not in recs or not recs[p["prompt_id"]]["parse_valid"])
    out["invalid_pct"] = 100 * inv / len(man)
    out["missing"] = missing
    for key, (lang, cond) in {"EN_P0": ("en", "canonical"), "TR_P0": ("tr", "canonical"), "TR_P3": ("tr", "P3")}.items():
        rows = [p for p in man if p["language"] == lang and p["condition"] == cond]
        out[key] = 100 * sum(1 for p in rows if p["prompt_id"] in recs and recs[p["prompt_id"]].get("correct")) / len(rows)
        for n in (4, 16, 64):
            rn = [p for p in rows if p["n"] == n]
            out[f"{key}_n{n}"] = 100 * sum(1 for p in rn if p["prompt_id"] in recs and recs[p["prompt_id"]].get("correct")) / len(rn)
    for key, left, right in (("d_P3_P0", ("tr", "canonical", "native_shared"), ("tr", "P3", "native_shared")),
                             ("d_TR_EN", ("en", "canonical", "native_shared"), ("tr", "canonical", "native_shared"))):
        pairs = _pairs(man, recs, left, right)
        b = bootstrap(pairs, allplanned)["d"]
        out[key] = (b["estimate"], b["low"], b["high"], len(pairs))
    return out

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--direct", required=True)
    ap.add_argument("--structured", default=None)
    ap.add_argument("--output", default="results/followup/followup_summary.json")
    a = ap.parse_args()
    man = [json.loads(l) for l in MAN.open(encoding="utf-8")]
    ids = {r["prompt_id"] for r in man}
    results = [arm("direct", load(a.direct, ids), man)]
    if a.structured:
        results.append(arm("structured", load(a.structured, ids), man))
    Path(a.output).parent.mkdir(parents=True, exist_ok=True)
    Path(a.output).write_text(json.dumps(results, indent=1))
    for r in results:
        f = lambda t: f"{t[0]:+.3f} [{t[1]:+.3f}, {t[2]:+.3f}]"
        print(f"{r['arm']:>10}: invalid {r['invalid_pct']:.2f}% | EN P0 {r['EN_P0']:.1f} | TR P0 {r['TR_P0']:.1f} | "
              f"TR P3 {r['TR_P3']:.1f} | dP3-P0 {f(r['d_P3_P0'])} | dTR-EN {f(r['d_TR_EN'])}")
        print("            by n (TR P0/TR P3/EN P0): " + "; ".join(
            f"n={n}: {r[f'TR_P0_n{n}']:.1f}/{r[f'TR_P3_n{n}']:.1f}/{r[f'EN_P0_n{n}']:.1f}" for n in (4, 16, 64)))
main()


In [ ]:
import hashlib, os, subprocess
if not os.path.exists('data/manifests/followup_reasoning_manifest.jsonl'):
    subprocess.run(['python3','scripts/build_followup_manifest.py'], check=True)
sha=hashlib.sha256(open('data/manifests/followup_reasoning_manifest.jsonl','rb').read()).hexdigest()
print(sha)
assert sha=='89dde862db84447ec23a20302f1c4449f9504aed3f681abc04c92681b185deee', 'manifest differs from the one used in the manuscript'

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time
subprocess.Popen(['ollama','serve'], stdout=open('/content/ollama.log','w'), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull qwen2.5:14b
!curl -s localhost:11434/api/tags | python3 -c "import sys,json; [print(m['name'], m['digest'][:8]) for m in json.load(sys.stdin)['models']]"

Digest **7cdf5a01** olmalı (ana taramadaki 14B ağırlıkları). Farklıysa sonuçlar yine kullanılabilir ama makalede belirtilmesi gerekir.

In [ ]:
# Kısa deneme: 5 istem; raw_text sonunda 'FINAL: EVET/HAYIR' görünmeli
!PYTHONPATH=src python3 -m inference.ollama_reasoning_runner --manifest data/manifests/followup_reasoning_manifest.jsonl --model qwen2.5:14b --strategy structured_reasoning --output $OUT/qwen2-5-14b_structured.jsonl --limit 5 --progress-every 5
import json
recs=[json.loads(l) for l in open(OUT+'/qwen2-5-14b_structured.jsonl') if '"inference"' in l]
for r in recs[-5:]: print(r['parse_valid'], r['output_tokens'], repr(r['raw_text'][-60:]))

In [ ]:
# Tam koşu (kesilirse bu hücreyi yeniden çalıştırın; kaldığı yerden devam eder)
!PYTHONPATH=src python3 -m inference.ollama_reasoning_runner --manifest data/manifests/followup_reasoning_manifest.jsonl --model qwen2.5:14b --strategy structured_reasoning --output $OUT/qwen2-5-14b_structured.jsonl --progress-every 50

In [ ]:
!PYTHONPATH=src python3 scripts/analyze_followup.py --direct results/p1/qwen2-5-14b.jsonl --structured $OUT/qwen2-5-14b_structured.jsonl --output $OUT/followup_summary.json
from google.colab import files
files.download(OUT+'/followup_summary.json')
files.download(OUT+'/qwen2-5-14b_structured.jsonl')